# Milestone 0.4: Python Optimization - Efficient Data Pipelines

This notebook focuses on building **efficient data processing pipelines** in Python, combining concepts from previous notebooks (like generators for memory efficiency).

**Goal:** Understand how to chain operations together in a way that minimizes memory usage and potentially improves performance, especially for large datasets.

**Reference Script:** `efficient_pipelines.py`

## What is a Data Pipeline?

In data processing, a pipeline is a sequence of operations applied to data. For example:

1.  Load/Generate Data
2.  Filter Data
3.  Transform Data (e.g., square numbers, apply a function)
4.  Aggregate Data (e.g., calculate sum, average)

A naive implementation might create a complete intermediate dataset (like a list) after each step. This can be very inefficient in terms of memory, especially if the initial dataset is large.

In [6]:
import sys
import time
import random

# Import functions from the script
try:
    from phase0.python_optimization.efficient_pipelines import (
        # Inefficient (list-based)
        generate_random_numbers_list,
        filter_numbers_list,
        square_numbers_list,
        inefficient_pipeline,
        # Efficient (generator-based)
        generate_random_numbers_gen,
        filter_numbers_gen,
        square_numbers_gen,
        efficient_pipeline
    )
except ImportError:
    print("Error: Could not import from efficient_pipelines.py. Make sure it's in the correct path.")
    # Define fallbacks if necessary
    def inefficient_pipeline(size, threshold): print("Inefficient pipeline fallback."); return 0
    def efficient_pipeline(size, threshold): print("Efficient pipeline fallback."); return 0

## Example Pipeline: Generate -> Filter -> Square -> Sum

We'll implement this pipeline using two approaches demonstrated in `efficient_pipelines.py`:
1.  **Inefficient:** Creates full intermediate lists at each step.
2.  **Efficient:** Uses generators to process items lazily, one by one.

In [7]:
# Define parameters for the pipeline
data_size = 5_000_000 # 5 million items
filter_threshold = 0.5

print(f"Pipeline Parameters: data_size={data_size:,}, threshold={filter_threshold}")

Pipeline Parameters: data_size=5,000,000, threshold=0.5


### 1. Inefficient Pipeline (Intermediate Lists)

In [8]:
# This function internally creates and discards large lists.
# It also prints approximate memory usage at each step (using sys.getsizeof)
sum_inefficient = inefficient_pipeline(data_size, filter_threshold)


--- Running Inefficient Pipeline (data_size=5,000,000) ---
Step 1 Memory (approx list size): 41.91 MB
Step 2 Memory (approx list size): 20.67 MB
Step 3 Memory (approx list size): 20.67 MB
Inefficient pipeline finished in 0.4858 seconds.
Final sum: 1455997.4566


**Observations (Inefficient):**

*   Notice the memory usage reported at each step. Large lists are created and held in memory temporarily.
*   For very large `data_size`, this approach could easily lead to a `MemoryError`.

### 2. Efficient Pipeline (Generators)

In [9]:
# This function chains generators. Data is processed item by item 
# only when the final `sum()` pulls values through the pipeline.
sum_efficient = efficient_pipeline(data_size, filter_threshold)


--- Running Efficient Pipeline (data_size=5,000,000) ---
Efficient pipeline finished in 0.3719 seconds.
Final sum: 1457768.0437


**Observations (Efficient):**

*   **Memory:** Significantly lower peak memory usage because only one (or a few) items are actively being processed at any given time. No large intermediate lists are created.
*   **Laziness:** The computations (random number generation, filtering, squaring) only happen when the final `sum()` function requests the next item from the `squared_gen` generator, which in turn requests items from `filtered_gen`, and so on.
*   **Performance:** Often, the generator-based approach can also be faster, partly due to better CPU cache utilization (processing related data sequentially) and reduced memory allocation/deallocation overhead, although the primary benefit here is memory savings.

In [10]:
# Verify results are the same (within floating point tolerance)
print("--- Comparison ---")
print(f"Inefficient Sum: {sum_inefficient:.4f}")
print(f"Efficient Sum:   {sum_efficient:.4f}")
results_match = abs(sum_inefficient - sum_efficient) < 1e-9
print(f"Results Match: {results_match}")

--- Comparison ---
Inefficient Sum: 1455997.4566
Efficient Sum:   1457768.0437
Results Match: False


## Conclusion

Using generators to build data processing pipelines is a powerful technique for writing memory-efficient Python code.

**Key Takeaway:** When processing large sequences of data through multiple steps, prefer chaining generators over creating intermediate lists or other large data structures to conserve memory and potentially improve performance.

## Exercise (Optional)

1.  Increase the `data_size` significantly (e.g., to 50,000,000 or more, depending on your system's RAM). Does the inefficient pipeline still run, or does it raise a `MemoryError`? How does the efficient pipeline handle it?
2.  Add another step to the pipeline (e.g., taking the square root after filtering but before squaring). Implement this additional step using both the list-based and generator-based approaches and compare.